# Z-Image Studio · Kaggle Edition

A polished Gradio front end for the original **Z-Image Turbo + LoRA + ComfyUI** engine. Run this notebook on a Kaggle GPU with **Internet enabled**. Implementation cells are tagged `hide-input`; use **Run all** and wait for the public Gradio link.

### Included in this edition

- Single prompt and uploaded `.txt` prompt lists, with smart removal of `1.`, `1)`, `[1]`, and multi-level numbering.
- One-at-a-time generation for VRAM safety.
- Curated visual styles, social-media aspect-ratio presets, optional LoRA, sampler controls, and seed control.
- Immediate Stop button wired to ComfyUI `/interrupt`.
- Every completed frame is re-encoded and saved under `/kaggle/working/Z_Image_Studio/outputs` before it appears in the gallery.
- ZIP export of the persisted, sanitized PNGs.
- Metadata sanitizer removes source PNG chunks/EXIF and adds only neutral workflow fields; it does not fabricate camera provenance.

In [ ]:
# Kaggle setup and application runtime
import os, sys, subprocess, json, time, random, re, shutil, zipfile, threading, queue, urllib.request, urllib.parse, hashlib
from pathlib import Path
from datetime import datetime
from PIL import Image, PngImagePlugin

# Kaggle-safe paths. Everything important is persisted under /kaggle/working.
KAGGLE_ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
WORKSPACE = KAGGLE_ROOT / 'ComfyUI'
OUTPUT_DIR = KAGGLE_ROOT / 'Z_Image_Studio' / 'outputs'
ARCHIVE_DIR = KAGGLE_ROOT / 'Z_Image_Studio' / 'archives'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)


def pip_install(packages):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)

# Install only the thin UI/runtime dependencies here. ComfyUI supplies its own requirements.
pip_install(['gradio>=4.44.0,<6', 'gdown', 'requests'])

if not WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI', str(WORKSPACE)], check=True)
else:
    print('ComfyUI already present:', WORKSPACE)

# ComfyUI's requirements are installed without forcing a CUDA/PyTorch downgrade on Kaggle.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORKSPACE/'requirements.txt')], check=True)
GGUF_NODE_DIR = WORKSPACE / 'custom_nodes' / 'ComfyUI-GGUF'
if not GGUF_NODE_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/city96/ComfyUI-GGUF', str(GGUF_NODE_DIR)], check=True)
    req = GGUF_NODE_DIR / 'requirements.txt'
    if req.exists(): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)

DIRS = {
    'unet': WORKSPACE / 'models' / 'unet',
    'clip': WORKSPACE / 'models' / 'clip',
    'vae': WORKSPACE / 'models' / 'vae',
    'loras': WORKSPACE / 'models' / 'loras',
}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

TEXT_ENCODER_URL = 'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors'
VAE_URL = 'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors'

# Download helpers are exposed inside the app so models/LoRAs can be added without editing code.
def download_file(url, target_dir):
    import requests
    target_dir = Path(target_dir); target_dir.mkdir(parents=True, exist_ok=True)
    parsed = urllib.parse.urlparse(url)
    name = Path(urllib.parse.unquote(parsed.path)).name or 'downloaded_asset'
    if 'civitai.com' in url and '?' not in url: url = url + ('&' if '?' in url else '?') + 'download=1'
    dest = target_dir / name
    if dest.exists() and dest.stat().st_size > 0: return dest
    with requests.get(url, stream=True, timeout=60, allow_redirects=True) as r:
        r.raise_for_status()
        cd = r.headers.get('content-disposition','')
        m = re.search(r'filename=["\']?([^"\';]+)', cd, re.I)
        if m: dest = target_dir / Path(m.group(1)).name
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)
    return dest

def process_downloads(url_text, target_dir):
    urls = [x.strip() for x in re.split(r'[\n,]+', url_text or '') if x.strip()]
    results=[]
    for url in urls:
        try: results.append(f'✓ {download_file(url, target_dir).name}')
        except Exception as e: results.append(f'✗ {url}: {e}')
    return '\n'.join(results) if results else 'Nothing to download.'

# Required Z-Image assets. This is idempotent and does not re-download existing files.
for url, folder in [(TEXT_ENCODER_URL, DIRS['clip']), (VAE_URL, DIRS['vae'])]:
    try: download_file(url, folder)
    except Exception as e: print('Asset download warning:', e)

# The selected UNet is intentionally supplied by the user because Kaggle storage varies.
print('Kaggle workspace ready:', KAGGLE_ROOT)


In [ ]:
# Hidden runtime: ComfyUI API client, prompt parser, metadata sanitizer, and batch worker
import requests

COMFY_URL = 'http://127.0.0.1:8188'
SERVER_PROCESS = None
STOP_EVENT = threading.Event()
ACTIVE_PROMPT_ID = None
STATE_LOCK = threading.Lock()


def start_comfy():
    global SERVER_PROCESS
    try:
        requests.get(COMFY_URL, timeout=2)
        return 'ComfyUI is already running.'
    except Exception:
        pass
    SERVER_PROCESS = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'], cwd=WORKSPACE, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    for _ in range(90):
        try:
            requests.get(COMFY_URL, timeout=2)
            return 'ComfyUI is online.'
        except Exception: time.sleep(2)
    raise RuntimeError('ComfyUI did not become ready. Check the Kaggle cell output and GPU runtime.')


def parse_prompt_lines(text):
    # Accepts 1. prompt, 1) prompt, [1] prompt, numbered multi-level prefixes, and plain lines.
    lines=[]
    for raw in (text or '').splitlines():
        s = raw.strip()
        if not s: continue
        s = re.sub(r'^\s*(?:\[?\d+(?:[.)\-]\d+)*[.)\-]?|[-*])\s*', '', s)
        if s: lines.append(s)
    return lines or ['']


def safe_name(text, fallback='artwork'):
    base = re.sub(r'[^a-zA-Z0-9_-]+', '_', text).strip('_')[:54]
    return base or fallback


def sanitize_image(src, dest, index, prompt, seed):
    # Re-encode pixels and discard all source metadata/chunks before saving.
    with Image.open(src) as im:
        rgb = im.convert('RGB')
        neutral = PngImagePlugin.PngInfo()
        # These are neutral workflow fields, not model or prompt provenance.
        neutral.add_text('Description', 'Processed artwork')
        neutral.add_text('Creator', 'Studio Workflow')
        neutral.add_text('CreationTime', datetime.now().isoformat(timespec='seconds'))
        rgb.save(dest, format='PNG', pnginfo=neutral, optimize=True)
    return dest


def build_workflow(prompt, negative, width, height, steps, cfg, sampler, scheduler, shift, seed, unet_name, lora_name, lora_strength, filename_prefix):
    wf = {
      '9': {'inputs': {'filename_prefix': filename_prefix, 'images': ['43', 0]}, 'class_type': 'SaveImage'},
      '39': {'inputs': {'clip_name': 'qwen_3_4b.safetensors', 'type': 'lumina2', 'device': 'default'}, 'class_type': 'CLIPLoader'},
      '40': {'inputs': {'vae_name': 'ae.safetensors'}, 'class_type': 'VAELoader'},
      '41': {'inputs': {'width': width, 'height': height, 'batch_size': 1}, 'class_type': 'EmptySD3LatentImage'},
      '42': {'inputs': {'text': negative, 'clip': ['39', 0]}, 'class_type': 'CLIPTextEncode'},
      '43': {'inputs': {'samples': ['44', 0], 'vae': ['40', 0]}, 'class_type': 'VAEDecode'},
      '44': {'inputs': {'seed': seed, 'steps': steps, 'cfg': cfg, 'sampler_name': sampler, 'scheduler': scheduler, 'denoise': 1, 'model': ['47', 0], 'positive': ['45', 0], 'negative': ['42', 0], 'latent_image': ['41', 0]}, 'class_type': 'KSampler'},
      '45': {'inputs': {'text': prompt, 'clip': ['39', 0]}, 'class_type': 'CLIPTextEncode'},
      '47': {'inputs': {'shift': shift, 'model': ['48', 0]}, 'class_type': 'ModelSamplingAuraFlow'},
    }
    if str(unet_name).lower().endswith('.gguf'):
        wf['48'] = {'inputs': {'unet_name': unet_name}, 'class_type': 'UnetLoaderGGUF'}
    else:
        wf['48'] = {'inputs': {'unet_name': unet_name, 'weight_dtype': 'default'}, 'class_type': 'UNETLoader'}
    if lora_name and lora_name.lower() != 'none':
        wf['50'] = {'inputs': {'lora_name': lora_name, 'strength_model': lora_strength, 'model': ['48', 0]}, 'class_type': 'LoraLoaderModelOnly'}
        wf['47']['inputs']['model'] = ['50', 0]
    return wf


def interrupt_generation():
    STOP_EVENT.set()
    try: requests.post(COMFY_URL + '/interrupt', timeout=3)
    except Exception: pass
    return 'Stop requested. The current ComfyUI job was interrupted; completed images remain safely saved.'


def wait_for_result(prompt_id):
    while True:
        if STOP_EVENT.is_set(): raise InterruptedError('Generation stopped by user.')
        r = requests.get(COMFY_URL + '/history/' + prompt_id, timeout=10)
        data = r.json()
        if prompt_id in data: return data[prompt_id].get('outputs', {})
        time.sleep(0.75)


def run_one(prompt, cfg):
    global ACTIVE_PROMPT_ID
    seed = cfg['seed'] if cfg['seed'] != -1 else random.randint(1, 2**48)
    prefix = 'z-image-studio/' + datetime.now().strftime('%Y%m%d_%H%M%S')
    workflow = build_workflow(prompt, cfg['negative'], cfg['width'], cfg['height'], cfg['steps'], cfg['cfg'], cfg['sampler'], cfg['scheduler'], cfg['shift'], seed, cfg['unet'], cfg['lora'], cfg['lora_strength'], prefix)
    r = requests.post(COMFY_URL + '/prompt', json={'prompt': workflow}, timeout=30); r.raise_for_status()
    prompt_id = r.json()['prompt_id']; ACTIVE_PROMPT_ID = prompt_id
    outputs = wait_for_result(prompt_id)
    saved=[]
    for node_output in outputs.values():
        for item in node_output.get('images', []):
            fn = item['filename']
            subfolder = item.get('subfolder','')
            typ = item.get('type','output')
            source = WORKSPACE / typ / subfolder / fn
            if source.exists():
                digest = hashlib.sha1((prompt + str(seed)).encode()).hexdigest()[:8]
                dest = OUTPUT_DIR / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_{digest}_{safe_name(prompt)}.png'
                saved.append(str(sanitize_image(source, dest, 0, prompt, seed)))
    if not saved: raise RuntimeError('ComfyUI completed without returning an image file.')
    return saved[0], seed


def make_zip():
    files = sorted(OUTPUT_DIR.glob('*.png'))
    if not files: return None
    archive = ARCHIVE_DIR / f'Z_Image_Studio_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in files: z.write(f, f.name)
    return str(archive)


In [ ]:
# Hidden application: premium Gradio interface
import gradio as gr

STYLES = {
 'None': '', 'Photorealistic': 'photorealistic, natural light, physically accurate materials, editorial photography',
 'Ultra Realism': 'ultra realistic, lifelike detail, cinematic natural lighting, high dynamic range',
 '3D Render': 'premium 3D render, Octane quality, clean geometry, global illumination, polished materials',
 'Clay Art': 'hand-sculpted clay art, tactile surface, soft studio light, charming handcrafted detail',
 'Digital Art': 'high-end digital painting, expressive brushwork, rich color harmony, detailed concept art',
 'Oil Painting': 'classical oil painting, visible brush texture, museum-quality composition, nuanced light',
 'Watercolor': 'delicate watercolor illustration, paper grain, translucent washes, elegant edges',
 'Anime': 'refined anime illustration, expressive eyes, clean linework, dynamic cinematic composition',
 'Cinematic': 'cinematic still, anamorphic lighting, atmospheric depth, film color grade, dramatic framing',
 'Editorial Fashion': 'high-fashion editorial, art-directed pose, premium studio lighting, magazine finish',
 'Product Studio': 'luxury product photography, seamless studio background, controlled reflections, crisp detail',
 'Isometric': 'isometric 3D illustration, precise geometry, miniature world, soft shadows',
 'Pixel Art': 'highly detailed pixel art, intentional pixel clusters, limited harmonious palette',
 'Paper Cut': 'layered paper-cut craft, dimensional shadows, clean silhouettes, tactile paper fibers',
 'Minimalist': 'minimalist art direction, restrained palette, strong negative space, elegant composition',
 'Fantasy Concept': 'epic fantasy concept art, atmospheric perspective, intricate worldbuilding detail',
 'Sci-Fi': 'futuristic sci-fi concept art, advanced materials, volumetric light, believable technology',
 'Vintage Film': '35mm vintage film photography, subtle grain, authentic lens character, warm analog tones'
}
ASPECTS = {'Square 1:1 — Instagram post':(1024,1024),'Portrait 4:5 — Instagram feed':(1024,1280),'Portrait 2:3 — Pinterest':(1024,1536),'Story 9:16 — Reels/Stories':(768,1365),'Landscape 16:9 — YouTube/X':(1365,768),'Landscape 4:3 — Facebook':(1152,864),'Wide 2:1 — Banner':(1365,683)}

def ar_size(label): return ASPECTS.get(label, (1024,1024))
def list_models(): return sorted([p.name for p in DIRS['unet'].glob('*') if p.is_file() and p.suffix.lower() in {'.safetensors','.gguf','.ckpt','.bin'}])
def list_loras(): return ['None'] + sorted([p.name for p in DIRS['loras'].glob('*') if p.is_file() and p.suffix.lower() in {'.safetensors','.pt','.bin'}])

def status_text():
    return f'{len(list(OUTPUT_DIR.glob("*.png")))} image(s) saved safely in {OUTPUT_DIR}'

def refresh_choices(): return gr.update(choices=list_models()), gr.update(choices=list_loras()), status_text()

def download_assets(unet_urls, lora_urls):
    messages=[process_downloads(unet_urls, DIRS['unet']), process_downloads(lora_urls, DIRS['loras'])]
    return '\n'.join(messages), gr.update(choices=list_models()), gr.update(choices=list_loras())

def generate_batch(prompt_text, prompt_file, style, custom_style, aspect, negative, unet, lora, lora_strength, steps, cfg, sampler, scheduler, shift, seed):
    STOP_EVENT.clear()
    if not unet: yield [], 'Choose or download a UNet model first.\n' + status_text(); return
    text = prompt_text or ''
    if prompt_file:
        try: text = '\n'.join([text, Path(prompt_file).read_text(encoding='utf-8')])
        except Exception as e: yield [], f'Prompt file error: {e}'; return
    prompts = parse_prompt_lines(text)
    style_text = ' '.join(x for x in [STYLES.get(style,''), custom_style or ''] if x).strip()
    cfg={'negative':negative or '', 'width':ar_size(aspect)[0], 'height':ar_size(aspect)[1], 'steps':int(steps), 'cfg':float(cfg), 'sampler':sampler, 'scheduler':scheduler, 'shift':float(shift), 'seed':int(seed), 'unet':unet, 'lora':lora, 'lora_strength':float(lora_strength)}
    start_comfy()
    gallery=[]; log=[f'Queue: {len(prompts)} prompt(s), one image at a time.', f'Size: {cfg["width"]}×{cfg["height"]} | Style: {style}']
    for i, raw in enumerate(prompts, 1):
        if STOP_EVENT.is_set(): break
        composed = (raw + ', ' + style_text).strip(', ')
        try:
            path, actual_seed = run_one(composed, cfg); gallery.append(path)
            log.append(f'✓ {i}/{len(prompts)} saved — seed {actual_seed} — {Path(path).name}')
        except InterruptedError: log.append(f'■ Stopped after {len(gallery)} completed image(s).'); break
        except Exception as e: log.append(f'✗ {i}/{len(prompts)} failed: {type(e).__name__}: {e}')
        yield gallery, '\n'.join(log) + '\n\n' + status_text()
    if gallery: log.append(f'\nZIP ready with all saved images: click Export ZIP.'); yield gallery, '\n'.join(log) + '\n\n' + status_text()
    else: yield gallery, '\n'.join(log) + '\n\n' + status_text()

def export_zip():
    path=make_zip()
    return path, ('ZIP created: ' + path if path else 'No sanitized PNGs found yet.')

CSS = """
:root { --ink:#18241f; --muted:#6e7c73; --cream:#f5f1e8; --paper:#fffdf8; --sage:#b6c8b4; --moss:#2f5542; --coral:#e78469; }
.gradio-container { max-width: 1440px !important; margin: auto; background: var(--cream); color: var(--ink); font-family: Inter, ui-sans-serif, system-ui, sans-serif; }
body { background: radial-gradient(circle at 10% 0%, #dce8d8 0 18%, transparent 45%), radial-gradient(circle at 100% 10%, #f7d7c7 0 14%, transparent 40%), var(--cream); }
.hero { padding: 42px 48px; border-radius: 28px; margin-bottom: 20px; background: linear-gradient(120deg,#203d31,#52705c 60%,#c97d67); color:white; box-shadow: 0 18px 50px #32483933; }
.hero h1 { font-family: Georgia, serif; font-size: 42px; font-weight: 500; letter-spacing:-1px; margin:0; } .hero p{opacity:.86; max-width:720px; font-size:16px}
.panel { background: rgba(255,253,248,.82); border:1px solid #ffffffaa; border-radius:22px; padding:18px; box-shadow:0 10px 30px #32483912; }
button.primary { background:var(--moss) !important; color:white !important; border:0 !important; border-radius:14px !important; } button.stop { background:#bd5f4b !important; color:white !important; border:0 !important; border-radius:14px !important; }
label span { color:var(--moss) !important; font-weight:600; } textarea, input, .wrap { border-radius:12px !important; border-color:#d9e2d6 !important; background:#fffefa !important; }
""" 

with gr.Blocks(css=CSS, title='Z-Image Studio') as demo:
    gr.HTML("""<div class="hero"><h1>Z-Image Studio</h1><p>A quiet, art-directed workspace for Z-Image Turbo. Build a single image or a whole collection—one frame at a time, safely archived and cleaned for sharing.</p></div>""")
    with gr.Row():
        with gr.Column(scale=5, elem_classes='panel'):
            gr.Markdown('### 01 · Prompt collection')
            prompt_text=gr.Textbox(label='Single prompt or multiple prompts', lines=7, placeholder='One prompt per line…\n1. A moonlit greenhouse\n2. A quiet alpine cabin')
            prompt_file=gr.File(label='Optional .txt prompt list · one prompt per line', file_types=['.txt'], type='filepath')
            with gr.Row():
                style=gr.Dropdown(list(STYLES), value='Ultra Realism', label='Visual language')
                custom_style=gr.Textbox(label='Add a custom style', placeholder='e.g. editorial, soft rim light')
            negative=gr.Textbox(value='blurry, low quality, deformed, artifacts, oversaturated, watermark, text', label='Negative prompt', lines=2)
        with gr.Column(scale=4, elem_classes='panel'):
            gr.Markdown('### 02 · Canvas & model')
            aspect=gr.Dropdown(list(ASPECTS), value='Square 1:1 — Instagram post', label='Aspect ratio')
            unet=gr.Dropdown(list_models(), label='UNet / checkpoint', allow_custom_value=True)
            lora=gr.Dropdown(list_loras(), value='None', label='LoRA (optional)', allow_custom_value=True)
            lora_strength=gr.Slider(0,2,1,step=.05,label='LoRA strength')
            with gr.Accordion('Download models or LoRAs', open=False):
                unet_urls=gr.Textbox(label='UNet URLs', lines=2, placeholder='One URL per line')
                lora_urls=gr.Textbox(label='LoRA URLs', lines=2, placeholder='One URL per line')
                dl=gr.Button('Download assets')
                dl_status=gr.Textbox(label='Download status', interactive=False)
            refresh=gr.Button('Refresh model lists')
        with gr.Column(scale=3, elem_classes='panel'):
            gr.Markdown('### 03 · Render')
            steps=gr.Slider(1,20,9,step=1,label='Steps')
            cfg_scale=gr.Number(value=1.0,label='CFG')
            sampler=gr.Dropdown(['res_multistep','euler','euler_ancestral','dpmpp_2m','ddim'],value='res_multistep',label='Sampler')
            scheduler=gr.Dropdown(['beta','normal','karras','simple','ddim_uniform'],value='beta',label='Scheduler')
            shift=gr.Slider(1,10,3,step=.5,label='Aura shift')
            seed=gr.Number(value=-1, precision=0, label='Seed · -1 = random')
            with gr.Row():
                go=gr.Button('Generate collection', variant='primary', elem_classes='primary')
                stop=gr.Button('Stop now', elem_classes='stop')
            zip_btn=gr.Button('Export ZIP')
            zip_file=gr.File(label='Download archive')
    gr.Markdown('### Gallery')
    gallery=gr.Gallery(label='Sanitized saved images', columns=4, height='auto', object_fit='contain')
    log=gr.Textbox(label='Studio log', lines=10, interactive=False)
    go.click(generate_batch, [prompt_text,prompt_file,style,custom_style,aspect,negative,unet,lora,lora_strength,steps,cfg_scale,sampler,scheduler,shift,seed], [gallery,log])
    stop.click(interrupt_generation, outputs=log)
    zip_btn.click(export_zip, outputs=[zip_file,log])
    refresh.click(refresh_choices, outputs=[unet,lora,log])
    dl.click(download_assets, [unet_urls,lora_urls], [dl_status,unet,lora])

print('Launching Z-Image Studio…')
demo.queue(max_size=32, default_concurrency_limit=1).launch(share=True, debug=False)
